# Transcript Analysis (Robust / Hypothesis-Generating)

In this notebook, I analyze my TikTok **transcripts** alongside basic performance counts to generate **testable hypotheses** about:

- which **formats** perform better (e.g., “goodnight to everyone except …” vs “goodnight to no one except …”),
- which **words / short phrases** show up disproportionately often in better-performing videos.

**Important:** with ~50 videos, results are **directional**. I’m treating these outputs as **ideas to A/B test**, not definitive proof.


## 1) Imports & settings

In [1]:
import os
import re
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", 120)

# Reduce spammy warnings from libraries (keep important ones)
warnings.filterwarnings("once")

## 2) Load data

This expects a CSV that includes at least:
- `video_id`
- `transcript`
- `view_count`
- (optionally) `like_count`, `comment_count`, `repost_count`
- (optionally) a posted timestamp column such as `posted_at_utc` or `posted_at_local`

In [2]:
# Try a few common locations (edit if needed)
CANDIDATE_PATHS = [
    Path("../data/derived/tiktok_manifest_with_transcripts.csv"),
    Path("..") / "data" / "derived" / "tiktok_manifest_with_transcripts.csv",
    Path("data/derived/tiktok_manifest_with_transcripts.csv"),
    Path("tiktok_manifest_with_transcripts.csv"),
]

path = None
for p in CANDIDATE_PATHS:
    if p.exists():
        path = p
        break

if path is None:
    raise FileNotFoundError(
        "Could not find tiktok_manifest_with_transcripts.csv. "
        "Update CANDIDATE_PATHS to point to your CSV."
    )

df_raw = pd.read_csv(path)
print("Loaded:", path.resolve())
print("Rows:", len(df_raw), "Cols:", len(df_raw.columns))
display(df_raw.head(3))

Loaded: /Users/Logan/Desktop/TikTok Goodnight/tiktok-analysis/data/derived/tiktok_manifest_with_transcripts.csv
Rows: 53 Cols: 15


,video_id,video_url,title,description,upload_date,timestamp,duration,view_count,like_count,comment_count,repost_count,posted_at_utc,posted_at_local,upload_date_dt,transcript
0,7594277449698004238,https://www.tiktok.com/@humbletoker/video/7594277449698004238,"Goodnight to everyone EXCEPT challange, level hard. Follow for more. ...","Goodnight to everyone EXCEPT challange, level hard. Follow for more. I’m single and lonely #goodnight #influencer",20260112,1768180540,64,656,28,3,0,2026-01-12 01:15:40+00:00,2026-01-11 20:15:40-05:00,2026-01-12,Good night to everyone except 99% of people lose so good luck Starting with a super specific category Anyone taller ...
1,7593500470837087501,https://www.tiktok.com/@humbletoker/video/7593500470837087501,TikTok video #7593500470837087501,NaN,20260109,1767999638,73,320,18,3,0,2026-01-09 23:00:38+00:00,2026-01-09 18:00:38-05:00,2026-01-09,Good night to everyone except for the following people. This is the fast food edition and only 3% of people win so g...
2,7592046457117625613,https://www.tiktok.com/@humbletoker/video/7592046457117625613,This may be easier than 2% actually… goodnight to everyone who follow...,This may be easier than 2% actually… goodnight to everyone who follows me! #goodnighttoeveryoneexcept,20260106,1767661102,63,1339,53,9,63,2026-01-06 00:58:22+00:00,2026-01-05 19:58:22-05:00,2026-01-06,"Good night to everyone except for the following people, only 2% win so good luck. Starting with the broad category. ..."


In [ ]:
# --- Working copy for analysis ---
# I keep df_raw as the untouched input, and do all transforms on df.
df = df_raw.copy()

print("Rows:", len(df), "| Columns:", len(df.columns))
display(df.head(3))


## 3) Standardize types + engineer basic metrics

Fixes a common failure mode: **mixed timezones** when parsing datetimes.
We:
- parse datetimes with `utc=True` so everything becomes comparable,
- compute rate metrics (like/comment/repost per view),
- optionally compute `views_per_day` if timestamps exist.

In [3]:
# --- Standardize types + rate metrics (robust) ---
# This cell assumes df exists; if not, fall back to df_raw.
try:
    df
except NameError:
    df = df_raw.copy()

df = df.copy()
df["video_id"] = df["video_id"].astype(str)

# --- Datetimes: parse as UTC to avoid mixed-timezone errors ---
# Keep original columns, but also create parsed UTC versions for time math.
for col in ["posted_at_utc", "posted_at_local", "upload_date_dt"]:
    if col in df.columns:
        parsed = pd.to_datetime(df[col], errors="coerce", utc=True)  # always tz-aware UTC
        df[col + "_dt_utc"] = parsed

# Pick a "best available" posted time column for age calculations
posted_col = None
for candidate in ["posted_at_utc_dt_utc", "posted_at_local_dt_utc", "upload_date_dt_dt_utc"]:
    if candidate in df.columns and df[candidate].notna().any():
        posted_col = candidate
        break

# --- Numeric counts ---
for c in ["view_count", "like_count", "comment_count", "repost_count"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df["view_count"] = df["view_count"].fillna(0)

for c in ["like_count", "comment_count", "repost_count"]:
    if c in df.columns:
        df[c] = df[c].fillna(0)

# --- Rate metrics (protect against divide-by-zero) ---
denom = df["view_count"].replace(0, np.nan)
if "like_count" in df.columns:
    df["like_rate"] = df["like_count"] / denom
if "comment_count" in df.columns:
    df["comment_rate"] = df["comment_count"] / denom
if "repost_count" in df.columns:
    df["repost_rate"] = df["repost_count"] / denom

# --- Optional: views per day (controls for time since posting) ---
if posted_col is not None:
    now_utc = pd.Timestamp.now(tz="UTC")  # already tz-aware
    age_days = (now_utc - df[posted_col]).dt.total_seconds() / (3600 * 24)
    df["age_days"] = age_days
    df["views_per_day"] = df["view_count"] / df["age_days"].replace(0, np.nan)

# log scale views is often easier to model/plot
df["log_views"] = np.log1p(df["view_count"])

# --- Transcript cleaning + simple text features ---
def clean_text(s: str) -> str:
    s = "" if pd.isna(s) else str(s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

df["transcript_clean"] = df["transcript"].map(clean_text)

df["n_chars"] = df["transcript_clean"].str.len()
df["n_words"] = df["transcript_clean"].str.split().map(len)

df["n_questions"] = df["transcript_clean"].str.count(r"\?")
df["anyone_who_count"] = df["transcript_clean"].str.lower().str.count(r"\banyone who\b")

display(df[[
    "video_id","view_count","like_count","comment_count","repost_count",
    "like_rate","comment_rate","repost_rate",
    "duration","n_words","n_questions","anyone_who_count"
]].head(5))


NameError: name 'df' is not defined

## 4) Transcript normalization (rule-based, no LLM)

Whisper-style transcription errors create fake “different phrases”:
- `good night` vs `goodnight`
- `know one` vs `no one`
- `accept` vs `except`
- `some what` vs `somewhat`
- `summer specific` → **somewhat specific** 

We normalize to reduce false splits and make pattern matching reliable.

In [ ]:
def normalize_text(s: str) -> str:
    s = "" if pd.isna(s) else str(s)
    s = s.lower()

    # --- targeted replacements (expand over time as you notice patterns) ---
    repl = {
        r"\bgood\s*night\b": "goodnight",
        r"\bknow\s*one\b": "no one",
        r"\bnoone\b": "no one",
        r"\bno\s*one\b": "no one",
        r"\bnobody\b": "no one",
        r"\baccept\b": "except",  # common ASR slip in this context
        r"\bsome\s*what\b": "somewhat",
        r"\bsummer\s+supcefic\b": "somewhat specific",
        r"\bsummer\s+specific\b": "somewhat specific",
        r"\bsomewhat\s+specfic\b": "somewhat specific",
    }
    for pat, out in repl.items():
        s = re.sub(pat, out, s)

    # Keep letters/numbers/spaces; drop punctuation
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

# Use transcript, fallback to title if transcript missing/empty
title_col = "title" if "title" in df.columns else ("Video title" if "Video title" in df.columns else None)

df["transcript_raw"] = df.get("transcript", "").fillna("").astype(str)
df["transcript_norm"] = df["transcript_raw"].map(normalize_text)

if title_col is not None:
    df["title_raw"] = df[title_col].fillna("").astype(str)
    df["title_norm"] = df["title_raw"].map(normalize_text)
else:
    df["title_raw"] = ""
    df["title_norm"] = ""

df["text_norm"] = np.where(df["transcript_norm"].str.len() > 0, df["transcript_norm"], df["title_norm"])

# Quick before/after examples
sample = df.loc[df["transcript_raw"].str.len() > 0, ["video_id","transcript_raw","transcript_norm"]].head(3)
display(sample)
print("Missing/empty transcripts:", int((df['transcript_norm'].str.len()==0).sum()), "of", len(df))

,video_id,transcript_raw,transcript_norm
0,7594277449698004238,Good night to everyone except 99% of people lose so good luck Starting with a super specific category Anyone taller ...,goodnight to everyone except 99 of people lose so good luck starting with a super specific category anyone taller th...
1,7593500470837087501,Good night to everyone except for the following people. This is the fast food edition and only 3% of people win so g...,goodnight to everyone except for the following people this is the fast food edition and only 3 of people win so good...
2,7592046457117625613,"Good night to everyone except for the following people, only 2% win so good luck. Starting with the broad category. ...",goodnight to everyone except for the following people only 2 win so good luck starting with the broad category anyon...


Missing/empty transcripts: 0 of 53


## 5) Data quality checks (prevents misleading NaNs)

We check:
- transcript length,
- placeholders like `tiktok video #...`,
- missingness,
- and we refuse to run “group comparison” tests if groups are too small.

In [ ]:
# Basic transcript length features
df["n_words"] = df["text_norm"].str.split().map(len)
df["n_chars"] = df["text_norm"].str.len()

print("Transcript words — describe:")
display(df["n_words"].describe())

# Flag obvious placeholder transcripts
placeholder_pat = re.compile(r"^tiktok\s+video\s+\d+$", re.I)
df["is_placeholder"] = df["text_norm"].str.match(placeholder_pat, na=False)

print("Placeholder transcripts:", int(df["is_placeholder"].sum()))
print("Very short transcripts (<5 words):", int((df["n_words"] < 5).sum()))

# Optionally exclude placeholders from language analyses (keep them for metric-only analyses)
df_lang = df.loc[~df["is_placeholder"] & (df["n_words"] >= 5)].copy()
print("Rows kept for language analysis:", len(df_lang), "of", len(df))

Transcript words — describe:


count     53.000000
mean     209.301887
std       43.416846
min       60.000000
25%      176.000000
50%      211.000000
75%      240.000000
max      302.000000
Name: n_words, dtype: float64

Placeholder transcripts: 0
Very short transcripts (<5 words): 0
Rows kept for language analysis: 53 of 53


## 6) Format A/B test: “everyone except” vs “no one except”

We classify videos into:
- `everyone_except` if they contain “goodnight to everyone except”
- `noone_except` if they contain “goodnight to no one except” (includes “no one”, “nobody”, “know one” after normalization)
- `other` otherwise

Key robustness changes:
- match **anywhere** (not just at the beginning)
- normalization handles `good night`/`goodnight`, `know one`, `accept`
- fallback to title if transcript is empty

In [ ]:
# Patterns on normalized text
pat_everyone = re.compile(r"\bgoodnight\s+to\s+everyone\s+except\b")
pat_noone = re.compile(r"\bgoodnight\s+to\s+(no\s+one)\s+except\b")  # after normalization all variants -> 'no one'

t = df["text_norm"].fillna("")

df["format_group"] = np.select(
    [
        t.str.contains(pat_everyone, na=False),
        t.str.contains(pat_noone, na=False),
    ],
    ["everyone_except", "noone_except"],
    default="other",
)

display(df["format_group"].value_counts())

# Outcome metric choices
# Prefer views_per_day if available; otherwise view_count; also compare repost_rate as engagement proxy
outcomes = ["view_count", "views_per_day", "repost_rate", "comment_rate", "like_rate"]
outcomes = [c for c in outcomes if c in df.columns]

# Summarize by group with guardrails
g = df.groupby("format_group", dropna=False)

summary = g[outcomes].agg(["count","mean","median"])
display(summary)

# Guardrail: don't claim anything if groups are tiny
min_n = g.size().min()
print("Min group size:", int(min_n))
if min_n < 5:
    print("⚠️ One or more groups have <5 videos. Treat comparisons as VERY tentative.")

/var/folders/64/4spprg_x3rbcsx6lq98tf5yw0000gp/T/ipykernel_16789/1466739492.py:10: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  t.str.contains(pat_noone, na=False),


format_group
noone_except       30
everyone_except    15
other               8
Name: count, dtype: int64

view_count                       views_per_day              \
                     count          mean  median         count        mean   
format_group                                                                 
everyone_except         15   2374.933333  1393.0            15   14.293578   
noone_except            30  71215.666667  4444.5            30  290.734575   
other                    8   6272.125000  1689.0             8   25.733080   

                           repost_rate                     comment_rate  \
                    median       count      mean    median        count   
format_group                                                              
everyone_except  12.746538          15  0.026818  0.038593           15   
noone_except     18.996787          30  0.053816  0.057739           30   
other             6.986835           8  0.025678  0.022910            8   

                                    like_rate                      
                     mean    median     count      mean    median  
format_group                                                       
everyone_except  0.006080  0.005059        15  0.034385  0.032605  
noone_except     0.005125  0.004141        30  0.028254  0.027236  
other            0.005181  0.006088         8  0.028359  0.029531

Min group size: 8


## 7) Transcript language patterns (Top vs Bottom performers)

We look for words / short phrases that show up more often in your best-performing videos than in your worst-performing videos.

**Goal:** generate actionable scripting ideas (hypotheses) like:
- phrases common in winners,
- language patterns associated with higher views or repost behavior,
- what to A/B test next.

**Method (robust / small-data friendly):**
- Choose a target metric (default: `view_count` or `views_per_day` if available).
- Split videos into **Top** and **Bottom** buckets (top 25% vs bottom 25%).
- Build unigram + bigram features from normalized transcripts.
- Compute **lift** with minimum support:
  - `top_rate`, `bottom_rate`, `lift = top_rate / bottom_rate`
  - require phrase appears in at least `min_support` videos overall to avoid one-offs.

In [ ]:
# --- Section 7: Transcript language patterns (Top vs Bottom performers) ---
# Robust to sparse matrices + avoids index/mask misalignment.

from sklearn.feature_extraction.text import CountVectorizer

# Choose metric to rank by
target = "view_count"   # change to "repost_rate" / "comment_rate" etc if you want
q = 0.25                # top/bottom quartiles
min_support = 2         # only show phrases that appear in >= this many videos

# Make sure transcript text exists + index is clean (critical!)
df = df.copy().reset_index(drop=True)
df["transcript_clean"] = df["transcript_clean"].fillna("").astype(str)

metric = pd.to_numeric(df[target], errors="coerce")
valid = metric.notna() & (df["transcript_clean"].str.len() > 0)

df_s = df.loc[valid].copy().reset_index(drop=True)
metric_s = pd.to_numeric(df_s[target], errors="coerce")

if len(df_s) < 8:
    print(f"Not enough rows for top/bottom split after filtering: n={len(df_s)}")
else:
    lo = metric_s.quantile(q)
    hi = metric_s.quantile(1 - q)

    top_mask = metric_s >= hi
    bot_mask = metric_s <= lo

    top_idx = np.flatnonzero(top_mask.to_numpy())
    bot_idx = np.flatnonzero(bot_mask.to_numpy())

    print(f"Using target={target} | top n={len(top_idx)} | bottom n={len(bot_idx)} | total used n={len(df_s)}")

    if len(top_idx) == 0 or len(bot_idx) == 0:
        print("Top or bottom bucket is empty. Try a different metric or a smaller q (e.g., q=0.2).")
    else:
        vec = CountVectorizer(
            ngram_range=(1, 2),
            min_df=1,
            stop_words="english"
        )
        X = vec.fit_transform(df_s["transcript_clean"])
        vocab = np.array(vec.get_feature_names_out())

        X_top = X[top_idx, :]
        X_bot = X[bot_idx, :]

        # rates = fraction of docs in bucket containing the phrase at least once
        top_rate = np.asarray(X_top.getnnz(axis=0)).ravel() / len(top_idx)
        bot_rate = np.asarray(X_bot.getnnz(axis=0)).ravel() / len(bot_idx)

        support = np.asarray(X.getnnz(axis=0)).ravel()

        # Lift with smoothing to avoid infinities
        eps = 1e-6
        lift_top = (top_rate + eps) / (bot_rate + eps)
        lift_bot = (bot_rate + eps) / (top_rate + eps)

        out = pd.DataFrame({
            "phrase": vocab,
            "support": support,
            "top_rate": top_rate,
            "bottom_rate": bot_rate,
            "lift_top": lift_top,
            "lift_bottom": lift_bot,
        })

        out = out[out["support"] >= min_support].sort_values("lift_top", ascending=False)

        print("\nPhrases disproportionately common in TOP bucket (hypothesis ideas):")
        display(out.head(25))

        out2 = out.sort_values("lift_bottom", ascending=False)
        print("\nPhrases disproportionately common in BOTTOM bucket (avoid / rethink ideas):")
        display(out2.head(25))


Target: views_per_day
Top bucket: 14 Bottom bucket: 14

Phrases disproportionately common in TOP bucket (hypotheses):


,phrase,support,top_rate,bottom_rate,lift
284,birthday as,8,0.571429,0.0,5.714286e+08
439,comment it,8,0.571429,0.0,5.714286e+08
962,it below,8,0.571429,0.0,5.714286e+08
1570,same birthday,8,0.571429,0.0,5.714286e+08
276,below and,7,0.500000,0.0,5.000000e+08
1101,ll get,6,0.428571,0.0,4.285714e+08
1195,more if,6,0.428571,0.0,4.285714e+08
2108,what this,5,0.357143,0.0,3.571429e+08
191,as the,4,0.285714,0.0,2.857143e+08
447,comments,4,0.285714,0.0,2.857143e+08



Phrases disproportionately common in BOTTOM bucket (avoid / rethink):


,phrase,support,top_rate,bottom_rate,lift_bottom
795,goodnight and,7,0.0,0.500000,5.000000e+08
1106,lose,7,0.0,0.500000,5.000000e+08
1409,people will,7,0.0,0.500000,5.000000e+08
2208,will lose,7,0.0,0.500000,5.000000e+08
92,all,5,0.0,0.357143,3.571429e+08
1925,time,5,0.0,0.357143,3.571429e+08
2228,with two,5,0.0,0.357143,3.571429e+08
142,and we,4,0.0,0.285714,2.857143e+08
847,has dog,4,0.0,0.285714,2.857143e+08
1107,lose and,4,0.0,0.285714,2.857143e+08


## 8) Optional: bootstrap difference in means (small-sample friendly)

Instead of p-values, we estimate a confidence interval for the difference between groups.
This is still not “proof,” but it’s more honest under small sample sizes.

In [ ]:
def bootstrap_diff(a, b, n=5000, seed=0):
    rng = np.random.default_rng(seed)
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    a = a[np.isfinite(a)]
    b = b[np.isfinite(b)]
    if len(a) < 3 or len(b) < 3:
        return None
    diffs = []
    for _ in range(n):
        aa = rng.choice(a, size=len(a), replace=True)
        bb = rng.choice(b, size=len(b), replace=True)
        diffs.append(np.mean(aa) - np.mean(bb))
    diffs = np.asarray(diffs)
    return float(np.mean(diffs)), float(np.quantile(diffs, 0.05)), float(np.quantile(diffs, 0.95))

metric = "views_per_day" if df["views_per_day"].notna().any() else "view_count"

a = df.loc[df["format_group"]=="everyone_except", metric]
b = df.loc[df["format_group"]=="noone_except", metric]

out = bootstrap_diff(a, b)
print("Metric:", metric)
if out is None:
    print("Not enough data in both groups for bootstrap CI.")
else:
    mean_diff, lo, hi = out
    print(f"Mean(top-everyone_except) - Mean(noone_except): {mean_diff:,.3f}")
    print(f"Approx 90% bootstrap CI: [{lo:,.3f}, {hi:,.3f}]")

## 9) Next actions (how to turn this into growth)

Use outputs as **hypotheses**:
- Pick 1–2 language patterns / hooks from Section 7.
- Create 6–10 videos where you keep everything else similar and vary only the hook/phrase.
- Track `repost_rate` and `comment_rate` as primary success metrics (views are confounded).
- Re-run this notebook after you add those new videos.

If you want, we can add a “tracking sheet” section that writes a CSV of the next planned experiments.